> **Notebook usage note:** `retail.csv` must be in the same folder as the notebook. The `.py` version supports a command-line path, but in Jupyter the kernel's `sys.argv` is not a Python file path, so this notebook is set up to read `"retail.csv"` directly. Do not confuse the study-purpose MSE values/theory written in this notebook with measured results; the final table uses the actual run results in `results/hw2_retail_results.csv`.

# Homework 2, Problem 2(b)–(e) — Understanding It Line by Line from Scratch

This notebook is **not a file for simply copying code and submitting it**; it is a study file for understanding the code all the way down to where the math meets the real data. In each section, **read the plain-language explanation first**, then **run the code while following the English comments line by line**.

**Setup:** Put `retail.csv` in the same folder. Required packages: `numpy`, `pandas`, `scikit-learn`, `torch`.

**Chosen design**: Train/Validation/Test = 70/15/15; 5-fold OOF Target Encoding; log-quantity MSE; Ridge alpha chosen on validation; 8-D embedding each; Hybrid MLP; Early Stopping; a common Test evaluation at the very end.

**Important:** The result numbers only appear when you actually run the code on the provided `retail.csv`. The three rows and numbers used for hand calculation below are a made-up example.

## Warm-up ① Why do we need matrices?

We want to predict $\hat t=b+w_1x_1+w_2x_2$ from two numbers $x_1,x_2$. Here are three made-up transactions.

| Row | $x_1$ | $x_2$ | Target $t$ |
|---|---:|---:|---:|
| 1 | 1 | 0 | 3 |
| 2 | 0 | 1 | 4 |
| 3 | 1 | 1 | 6 |

This data gives three equations: $b+w_1=3$, $b+w_2=4$, $b+w_1+w_2=6$. Solving by hand gives $b=1,w_1=2,w_2=3$.

In matrix form,

$$A=\begin{bmatrix}1&1&0\\1&0&1\\1&1&1\end{bmatrix},\quad
\beta=\begin{bmatrix}b\\w_1\\w_2\end{bmatrix},\quad
 t=\begin{bmatrix}3\\4\\6\end{bmatrix}.$$

Here each **row** is one transaction, and the 1 in the first **column** is the slot for the intercept $b$. `A @ beta` computes all three predictions at once.

Least squares is the method that minimizes the sum of squared errors $\|A\beta-t\|_2^2$. Ridge adds a penalty $\alpha(w_1^2+w_2^2)$ to it.

$$(A^\top A+\alpha D)\beta=A^\top t,\qquad D=\operatorname{diag}(0,1,1).$$

$A^\top A$ is the transpose of $A$ multiplied by $A$, and the 0 in $D$ means "do not penalize the intercept."

In [ ]:
# This cell is a hand-calculation check on 3 data points that runs even without retail.csv.
import numpy as np
A_demo = np.array([[1., 1., 0.], [1., 0., 1.], [1., 1., 1.]])  # first column is the 1 for the intercept
x_demo = np.array([3., 4., 6.])  # actual targets
D_demo = np.diag([0., 1., 1.])  # no penalty on b
beta_ols = np.linalg.solve(A_demo.T @ A_demo, A_demo.T @ x_demo)  # ordinary least squares
beta_ridge_demo = np.linalg.solve(A_demo.T @ A_demo + 1.0*D_demo, A_demo.T @ x_demo)  # Ridge with alpha=1
print('A^T A =\n', A_demo.T @ A_demo)
print('OLS  [b,w1,w2] =', beta_ols)  # [1,2,3]
print('Ridge[b,w1,w2] =', beta_ridge_demo)  # [3.5,0.375,0.875]


## Warm-up ② Why should we scale the inputs?

The same person's age can be written as **12 years** or **144 months**. $3\times12=36$ and $0.25\times144=36$ give exactly the same prediction. But the Ridge penalty differs: $3^2=9$ versus $0.25^2=0.0625$.

**We only changed the unit of the number, yet the penalty changed by a factor of 144.** That is why we standardize inputs as $z=(x-\mu)/s$. Age 15 with mean 12 and standard deviation 3 gives $z=1$, and 180 months with mean 144 months and standard deviation 36 months also gives $z=1$.

In addition, if input magnitudes differ extremely, the conditioning of $A^\top A$ can get worse. Standardization helps stabilize the computation, but it cannot completely eliminate the problem of two columns that are too highly correlated.

In [ ]:
age_years = 12             # write the age in 'years'
age_months = 12 * age_years  # write the same age in 'months'
print(3 * age_years, 0.25 * age_months)  # both predictions are 36
print(3**2, 0.25**2)  # same prediction, but the Ridge penalties are 9 and 0.0625
print((15 - 12)/3, (180 - 144)/36)  # after standardization both are 1


---
## Now the actual code for Homework 2(b)–(e)

Each cell below is a self-contained **explanation + runnable code** pair. Every key statement in a cell has a `#` comment in English. Run one cell at a time with **Shift+Enter**.

> **Notebook path note:** Below, the code has been changed to read `retail.csv` from the same folder as the notebook. If the CSV is in a different folder, edit just the one `file_path` line.

## 1. Libraries: importing data tables, matrices, neural networks, and preprocessing tools

**Goal:** start with what `import` is. Python brings in tools for each kind of task from *libraries*.

- `numpy`: computations on long lists of numbers/matrices. `@` is matrix multiplication, `**2` is squaring.
- `pandas`: tidying data, e.g. per-`StockCode` means from the transaction table.
- `torch`: trainable embeddings and neural networks, automatic differentiation, GPU or CPU.
- `sklearn`: KFold, input standardization, and a reference Ridge to check our answer.

**Beginner question:** In `import numpy as np`, `as np` is a short nickname for the library. `from torch import nn` picks out the neural-network building blocks inside PyTorch. **Read the `#` comment right next to every statement in order.**

In [ ]:
import sys  # used to read the 'retail.csv' file path given on the command line.
import copy  # 'copies' and saves the weights of the best-performing neural network.
import numpy as np  # numerical computation: matrix A, vector beta, means, squares, etc.
import pandas as pd  # reads the CSV as a table (DataFrame) and computes per-product/per-customer means.
import torch  # used for PyTorch Tensors, CPU/GPU computation, and automatic differentiation.
from torch import nn  # imports neural-network building blocks such as nn.Embedding, nn.Linear, nn.ReLU.
from torch.utils.data import TensorDataset, DataLoader  # feeds the data in small chunks (batches).
from sklearn.model_selection import KFold  # 5-fold split that reduces Target Encoding's leakage of a row's own target.
from sklearn.preprocessing import StandardScaler  # standardizes inputs by subtracting the mean and dividing by the standard deviation.
from sklearn.linear_model import Ridge  # checks whether our hand-built Ridge gives the same answer.


## 2. Experiment setup and fixing random numbers for reproducibility

**Why fix the seed?** Because we split the data randomly and initialize the network's starting numbers randomly, the score can change slightly on every run. Fixing `SEED=42` improves reproducibility.

`DEVICE` uses CUDA if a GPU is available, otherwise the CPU. Here `cuda` is the backend for NVIDIA GPUs. *Not having a GPU does not mean the code is wrong*.

In [ ]:
SEED = 42  # the starting number that makes 'randomness' exactly reproducible.
np.random.seed(SEED)  # makes NumPy's row shuffling identical on every run.
torch.manual_seed(SEED)  # fixes PyTorch's random initial embeddings/weights.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU if a CUDA GPU is available, otherwise CPU.
COLUMNS = ["StockCode", "CustomerID"]  # the two categorical input variables used to predict Quantity.
print("Compute device:", DEVICE)  # prints 'cuda' or 'cpu' to confirm which device is used.


## 3. Read the CSV → check for broken rows → one data split shared by all models

**Imagine 100 made-up transactions.** 70 go to Train, 15 to Validation, and 15 to Test.

- Train: learn the coefficients and embeddings.
- Validation: choose which `alpha` is best and how many epochs to train.
- Test: evaluate exactly once, at the very end, after all selection and training are done.

In `df.iloc[tr_idx]`, `iloc` is the tool that *selects table rows by integer row position*. `reset_index(drop=True)` is the step that renumbers the selected rows as 0,1,2,.... If `Quantity` is missing or is 0 or less, the code makes you re-check the assignment's assumptions.

**Note:** The split does not shuffle by category; it randomly divides **transaction rows**. Keep in mind that this serves a different purpose from evaluating generalization to unseen customers.

In [ ]:
file_path = "retail.csv"  # In the notebook, point directly to the CSV in the current folder instead of using argv.
df = pd.read_csv(file_path, dtype={c: "string" for c in COLUMNS})  # read product/customer IDs as strings to preserve leading zeros.
df = df.dropna(subset=COLUMNS + ["Quantity"]).reset_index(drop=True)  # drop rows missing an input or the target, then renumber the rows.
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="raise")  # convert quantity to numbers; if there is an 'abc', raise an error instead of silently skipping it.

# These checks report problems immediately so that we never silently train on bad data.
if not np.isfinite(df["Quantity"].to_numpy(dtype=float)).all():  # is there an infinity mixed into the quantities?
    raise ValueError("Quantity must be a finite number.")  # if so, do not train; report the cause instead.
if (df["Quantity"] <= 0).any():  # the assignment states Quantity is a 'positive integer'. Is anything 0 or less?
    raise ValueError("In this assignment, Quantity must be positive.")  # data that violates the assumption is checked first.
if len(df) < 10:  # check whether the file is too small to form 5 folds.
    raise ValueError("Too little data. Please use at least 10 rows.")

idx = np.random.permutation(len(df))  # randomly shuffle the row numbers 0..N-1; the rows themselves are not reordered yet.
cut_train = int(0.70 * len(df))  # the 70% point of the data: end position of the training data.
cut_valid = int(0.85 * len(df))  # the 85% point of the data: end position of the validation data.
tr_idx, va_idx, te_idx = np.split(idx, [cut_train, cut_valid])  # split into 70%, 15%, 15% groups of row numbers.
train = df.iloc[tr_idx].reset_index(drop=True)  # use the row numbers to pull out only the training data.
valid = df.iloc[va_idx].reset_index(drop=True)  # validation data: used to choose alpha and Early Stopping.
test = df.iloc[te_idx].reset_index(drop=True)  # test data: evaluated only after all model selection is finished.
print("Rows per split (train/valid/test):", len(train), len(valid), len(test))  # confirm the split sizes are as intended.


## 4. (b) Target Encoding function: product/customer code → 'mean Quantity of that category'

**A very small Target Encoding example:**

| Product | Past Quantity |
|---|---:|
| A | 2 |
| A | 4 |
| A | 6 |
| B | 10 |

The mean for A is `(2+4+6)/3=4`, and the mean for B is `10`. So we can put 4 into rows with `StockCode='A'`.

Do not confuse the function's **two inputs**.
- `reference`: the data used to *compute* the means. Its target `Quantity` is read.
- `query`: the data the already-computed means are *attached to*. This table's `Quantity` is not read.

`groupby(col)['Quantity'].mean()` computes the mean for each name, and `map(means)` replaces each name with its mean. A brand-new name that was never seen before can produce `NaN`, so `fillna(fallback)` uses the overall training mean instead.

In [ ]:
def target_encode(reference, query):  # reference is the table used to compute means; query is the table to convert to numbers.
    fallback = reference["Quantity"].mean()  # new customers not seen in training are replaced by the overall training mean.
    features = np.empty((len(query), len(COLUMNS)), dtype=np.float64)  # empty float matrix: number of query rows × 2 input columns.
    for j, col in enumerate(COLUMNS):  # handle StockCode when j=0 and CustomerID when j=1, one at a time.
        means = reference.groupby(col)["Quantity"].mean()  # e.g., if product A's quantities are 2,4,6, then A→4.
        mapped = query[col].map(means)  # replace query's category names with the means computed above.
        mapped = mapped.fillna(fallback)  # fill categories absent from reference / NA encodings with the overall mean.
        features[:, j] = mapped.to_numpy(dtype=float)  # [:, j] means "store into the j-th column of every row".
    return features  # return the encoded matrix of shape (number of query rows, 2).


## 5. 5-Fold Out-of-Fold Target Encoding: never encode a row with its own target

**Why OOF (Out-of-Fold)?** If a customer appears only once in the training table with Quantity 1000, computing the customer mean including that row makes the customer's encoded value exactly the target, 1000. It is as if we 'showed the answer in advance' during training, which raises the risk of overfitting.

So we split into 5 folds, **compute the means from 4 folds and attach them only to the remaining fold.** Repeating this 5 times gives every training row an encoding built without its own target.

For validation/test, the means are computed **from the full Train set only** and then attached. The actual Test `Quantity` must never be used in the encoding computation, not even once.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)  # randomly split train into 5 folds.
te_train = np.empty((len(train), 2), dtype=float)  # empty 2-column matrix to hold each training row's encoded values.
for fitting, heldout in kf.split(train):  # in each iteration, 4 folds compute the means and 1 fold gets encoded.
    ref_rows = train.iloc[fitting]  # the 4 folds used to compute target means (excluding the current heldout fold).
    query_rows = train.iloc[heldout]  # the 1 fold whose own targets must not be included in the mean computation.
    te_train[heldout] = target_encode(ref_rows, query_rows)  # store the encoded results exactly at the original row positions.
te_valid = target_encode(train, valid)  # validation targets are never used in the mean computation.
te_test = target_encode(train, test)  # test targets are likewise never used in the mean computation.
print("Target Encoding matrix shapes:", te_train.shape, te_valid.shape, te_test.shape)


## 6. Handling the heavy-tailed quantities and input scale

**Log transformation and standardization are different things.**

- `log1p(x)` = $\ln(1+x)$; for example, a quantity of 1 becomes about 0.693 and 1000 becomes about 6.909. This reduces the tendency of large quantities to dominate the loss excessively.
- `StandardScaler` computes $z=(x-\mu)/s$. It makes the product-mean quantity and customer-mean quantity comparable even if their *magnitudes/units* differ.

**Why use `fit_transform` only on train?** `fit` is the step that learns the mean $\mu$ and standard deviation $s$. Recomputing the mean on Validation and Test would bring in distribution information from data the model has never seen, so both use `transform` only.

**Important:** The log transform applied to the target `y_train` and the log + standardization applied to the *input* `te_train` serve different purposes.

In [ ]:
# Example: Quantity=1 → log(2)≈0.693, 1000 → log(1001)≈6.909.
# log1p(x) is log(1+x). Applying MSE after the log transform reduces the dominance of a few large quantities.
y_train = np.log1p(train["Quantity"].to_numpy(dtype=float))  # training target: log(1+Quantity).
y_valid = np.log1p(valid["Quantity"].to_numpy(dtype=float))  # validation target: same log scale.
y_test = np.log1p(test["Quantity"].to_numpy(dtype=float))  # test target: used only for the final evaluation.
true_qty = test["Quantity"].to_numpy(dtype=float)  # also keep the original quantity units for computing MAE/RMSE.

scaler = StandardScaler()  # tool that turns each input column into (value - training mean)/training standard deviation.
X_train = scaler.fit_transform(np.log1p(te_train))  # OOF encoding → log1p → compute training mean/std, then standardize.
X_valid = scaler.transform(np.log1p(te_valid))  # transform validation inputs using only the already-computed training scale.
X_test = scaler.transform(np.log1p(te_test))  # must not re-fit on the test distribution's mean/standard deviation.
print("Train mean of the two standardized inputs:", X_train.mean(axis=0))  # each train column's mean is close to 0.
print("Train standard deviation of the two standardized inputs:", X_train.std(axis=0))  # each train column's standard deviation is about 1.


## 7. Implementing the Ridge formula directly: (A^T A + alpha D) beta = A^T t

### The moment the math you solved by hand becomes real code

The prediction for each row is $\hat t_i=b+w_1x_{i1}+w_2x_{i2}$. Prepending a column of 1s gives

$$A=\begin{bmatrix}1&x_{11}&x_{12}\\1&x_{21}&x_{22}\\\vdots&\vdots&\vdots\end{bmatrix},\quad\beta=\begin{bmatrix}b\\w_1\\w_2\end{bmatrix}.$$

Ridge minimizes $\sum_i(t_i-\hat t_i)^2 + \alpha(w_1^2+w_2^2)$. The intercept $b$ is not penalized, so $D=\operatorname{diag}(0,1,1)$.

Taking the derivative and setting it to 0 gives

$$(A^\top A+\alpha D)\beta=A^\top t.$$

**Let's connect the code and the math one-to-one.**
- `A = np.column_stack([ones, X])` → add a column of 1s for the intercept to all data.
- `D = np.diag([0,1,1])` → exclude only the intercept from the penalty.
- `A.T @ A + alpha * D` → the 3×3 matrix on the left.
- `A.T @ y` → the 3-dimensional vector on the right.
- `np.linalg.solve(left, right)` → solves for the 3 unknowns $b,w_1,w_2$ at once.

Solving the linear system with `solve` is usually numerically preferable to explicitly computing the inverse matrix and multiplying by it.

In [ ]:
# Model: predicted log quantity = b + w1*(product encoding value) + w2*(customer encoding value).
# Objective: sum_i (t_i - prediction_i)^2 + alpha*(w1^2 + w2^2). The intercept b is not penalized.
def manual_ridge(X, y, alpha):  # X is (N,2), y is (N,), alpha is the Ridge penalty strength.
    ones = np.ones(len(X))  # create N ones to multiply the intercept b.
    A = np.column_stack([ones, X])  # prepend a column of 1s to X to build the (N,3) matrix A=[1,x1,x2].
    D = np.diag([0.0] + [1.0] * X.shape[1])  # D=diag(0,1,1): no penalty on the intercept b.
    left = A.T @ A + alpha * D  # left-hand 3×3 matrix of the normal equations: A^T A + alpha D.
    right = A.T @ y  # right-hand 3 numbers of the normal equations: A^T y.
    beta = np.linalg.solve(left, right)  # solve left @ beta = right to obtain [b,w1,w2].
    return beta  # return the three coefficients found.


## 8. Choosing alpha on Validation: Test must not be used for 'selection'

**`alpha` is not a coefficient learned directly from the target data.** It is a *hyperparameter*: the user defines candidates and picks the best one on Validation.

$\alpha=0$ is the same as ordinary least squares, and generally the larger $\alpha$ is, the larger the penalty on large coefficients. That does not mean a larger $\alpha$ is always better.

In `best_val_mse = float('inf')`, infinity is the initial value that guarantees the first candidate always beats the existing record. Passing the `if` statement stores the `alpha` with the smallest Validation MSE. Choosing `alpha` with Test can overestimate Test performance.

In [ ]:
alphas = [0.1, 1.0, 10.0, 100.0, 1000.0]  # compare from weak to strong penalties.
best_alpha = None  # no best alpha chosen yet.
best_val_mse = float("inf")  # the initial best score is infinity; the first candidate is guaranteed to be better.
for alpha in alphas:  # loop over each candidate alpha.
    beta_candidate = manual_ridge(X_train, y_train, alpha)  # learn coefficients from train only.
    A_valid = np.column_stack([np.ones(len(X_valid)), X_valid])  # also prepend the intercept column of 1s to the validation inputs.
    pred_valid = A_valid @ beta_candidate  # (N_valid,3)×(3,)→(N_valid,) vector of log predictions.
    val_mse = np.mean((y_valid - pred_valid) ** 2)  # mean squared log error on validation.
    print(f"alpha={alpha:g} / validation log-MSE={val_mse:.6f}")  # print each candidate's performance.
    if val_mse < best_val_mse:  # if this candidate's validation error is smaller than any so far,
        best_alpha = alpha  # remember the current alpha as the new best candidate.
        best_val_mse = val_mse  # also remember the current error to compare against the next candidate.

beta_ridge = manual_ridge(X_train, y_train, best_alpha)  # recompute the coefficients on the train data with the selected alpha.
print("Selected alpha:", best_alpha)  # show the best penalty, fixed before looking at Test at all.
print("Hand-implemented [b, w1, w2]:", beta_ridge)  # print the coefficients obtained by solving the normal equations.


## 9. Verifying the implementation against sklearn Ridge

**Why compare with sklearn?** Even if the derived formula is correct, a coding mistake can produce different coefficients. So we train once more under the same conditions with `sklearn.linear_model.Ridge`.

- `.intercept_` is $b$.
- `.coef_` is $[w_1,w_2]$.
- `np.r_[...]` concatenates the two arrays into one.
- `assert_allclose` checks that the numbers are equal while tolerating tiny floating-point differences.

For example, if `beta_ridge=[3.5,0.375,0.875]`, then `sk_beta` must have the same values within the tolerance.

In [ ]:
sk_model = Ridge(alpha=best_alpha)  # create scikit-learn's built-in Ridge model.
sk_model.fit(X_train, y_train)  # train with the same train data and the same alpha.
sk_beta = np.r_[sk_model.intercept_, sk_model.coef_]  # concatenate the intercept and the 2 coefficients into [b,w1,w2].
np.testing.assert_allclose(beta_ridge, sk_beta, rtol=1e-5, atol=1e-6)  # check they are equal within a small floating-point error.
print("NumPy hand-built Ridge == sklearn Ridge: PASS")  # if the check raised no exception, it succeeded.


## 10. Solving the same linear system in PyTorch to verify CPU/GPU Tensors

**What the GPU comparison really means:** We build PyTorch Tensors from the same $A$, same $D$, same $\alpha$, and same $t$ that we used in NumPy, and solve with `torch.linalg.solve`.

`dtype=torch.float64` means double-precision floating point. `device=DEVICE` runs the same computation on the GPU or CPU. `beta_torch.cpu().numpy()` moves the result back to a NumPy array to compare it with the earlier answer.

**Note:** A tiny system like $3\times3$ does not get faster on a GPU. This is *practice implementing the same formula in a different library*.

In [ ]:
A_np = np.column_stack([np.ones(len(X_train)), X_train])  # build the (N,3) training input A in NumPy.
A_torch = torch.tensor(A_np, dtype=torch.float64, device=DEVICE)  # convert the same A to a double Tensor and move it to the device.
y_torch = torch.tensor(y_train, dtype=torch.float64, device=DEVICE)  # put the log target y on the same device too.
d_values = torch.tensor([0., 1., 1.], dtype=torch.float64, device=DEVICE)  # penalty diagonal entries: [0,1,1].
D_torch = torch.diag(d_values)  # build the 3×3 diagonal matrix D.
left_torch = A_torch.T @ A_torch + best_alpha * D_torch  # the PyTorch formula is also A^T A + alpha D.
right_torch = A_torch.T @ y_torch  # compute the right-hand vector A^T y.
beta_torch = torch.linalg.solve(left_torch, right_torch)  # solve the same system of equations on the CPU or GPU.
np.testing.assert_allclose(beta_torch.cpu().numpy(), beta_ridge, rtol=1e-5, atol=1e-6)  # move Tensor values to CPU NumPy and compare.
print("NumPy Ridge == PyTorch Ridge: PASS")  # confirm the same normal equations give the same coefficients.


## 11. (c) Turning unique category names into integer IDs 1,2,3,...

**An embedding does not simply turn a label into a single real number.** Text such as `StockCode='A'` must first be converted to the integer ID 1 so that row 1 of the `nn.Embedding` table can be looked up.

Made-up example: `{'A':1, 'B':2, 'C':3}`. An unseen `D` is mapped to 0. Because `padding_idx=0` fixes embedding 0 to the zero vector, a new category that was never trained does not receive a random vector.

The vocabulary is built **from Train only**. Even if Validation and Test contain new categories, those names must not be added and learned.

In [ ]:
# An embedding takes integer IDs (1,2,3) instead of strings ('A123').
# ID 0 is reserved for 'new products/customers' not seen in train.
def make_ids(col):  # col is the string 'StockCode' or 'CustomerID'.
    vocabulary = {item: i + 1 for i, item in enumerate(train[col].unique())}  # build the ID dictionary from names in train only.
    def encode(series):  # this inner function converts the category names of the given column into IDs.
        mapped = series.map(vocabulary)  # names in the dictionary become 1,2,3,....
        filled = mapped.fillna(0)  # new names not in the dictionary are marked as 0.
        return filled.to_numpy(dtype=np.int64)  # convert to a 64-bit integer array suitable for nn.Embedding indices.
    return (  # return the ID arrays for train/valid/test together with the embedding table size.
        encode(train[col]),  # integer ID array for training categories.
        encode(valid[col]),  # integer ID array for validation categories.
        encode(test[col]),  # integer ID array for test categories.
        len(vocabulary) + 1,  # total number of IDs including 0 (unknown).
    )

stock_tr, stock_va, stock_te, n_stock = make_ids("StockCode")  # convert product codes to integer IDs.
cust_tr, cust_va, cust_te, n_cust = make_ids("CustomerID")  # convert customer codes to integer IDs.
cat_train = np.column_stack([stock_tr, cust_tr])  # each training row becomes [product ID, customer ID].
cat_valid = np.column_stack([stock_va, cust_va])  # build the validation input from the same two columns.
cat_test = np.column_stack([stock_te, cust_te])  # build the test input from the same two columns.
print("Unique products (+unknown):", n_stock, "/ unique customers (+unknown):", n_cust)


## 12. Network design shared by the Embedding model and the Hybrid model

**Let's check the model structure in numbers.**

- Product ID → 8 trainable numbers.
- Customer ID → a separate set of 8 trainable numbers.
- **(c)**: concatenated, 16 inputs → Linear(16,32) → ReLU → Linear(32,1).
- **(d)**: 16 from the two embeddings + 2 standardized Target Encodings = 18 inputs → same structure.

`__init__` **creates the layers to be trained**; `forward` defines **the order in which a batch actually passes through those layers**. `self.stock_emb(cat[:, 0])` looks up the ID as a vector of numbers. `torch.cat(...,dim=1)` joins the two vectors side by side, row by row.

**Caution on interpretation:** One dimension of an embedding does not necessarily mean something like 'product price'. It learns a latent representation needed for prediction.

In [ ]:
class RetailNet(nn.Module):  # inherits from nn.Module, the base class for PyTorch neural networks.
    def __init__(self, n_stock, n_cust, hybrid=False, emb_dim=8):  # constructor: table sizes, whether to combine, embedding dimension.
        super().__init__()  # lets the parent nn.Module perform its required internal initialization.
        self.hybrid = hybrid  # True means (d), False means (c); the same design is reused.
        self.stock_emb = nn.Embedding(n_stock, emb_dim, padding_idx=0)  # 8 numbers learned per product; unknown 0 is the zero vector.
        self.cust_emb = nn.Embedding(n_cust, emb_dim, padding_idx=0)  # 8 numbers learned per customer; unknown 0 is the zero vector.
        n_in = 2 * emb_dim + (2 if hybrid else 0)  # (c) has 8+8=16 inputs; (d) has 8+8+2=18 inputs.
        self.net = nn.Sequential(  # chains several neural-network operations in order.
            nn.Linear(n_in, 32),  # maps the 16/18 inputs to 32 trainable hidden values.
            nn.ReLU(),  # sets negatives to 0 and keeps positives, so nonlinear relationships can be learned.
            nn.Linear(32, 1),  # turns the 32 hidden values into a single 'predicted log quantity'.
        )

    def forward(self, cat, numeric):  # the computation path the actual data follows when the model is called.
        stock_ids = cat[:, 0]  # first column of every row: product integer ID.
        customer_ids = cat[:, 1]  # second column of every row: customer integer ID.
        stock_vector = self.stock_emb(stock_ids)  # look up the 8-number trainable vector for each product ID.
        customer_vector = self.cust_emb(customer_ids)  # look up the 8-number vector for each customer ID.
        parts = [stock_vector, customer_vector]  # list of the two embeddings to be concatenated.
        if self.hybrid:  # check whether this is the Hybrid model of (d).
            parts.append(numeric)  # if Hybrid, also add the 2 standardized Target Encodings to the list.
        x = torch.cat(parts, dim=1)  # join each row's inputs side by side to form 16 or 18 features.
        output = self.net(x)  # predict the log quantity via Linear→ReLU→Linear.
        return output.squeeze(1)  # remove the unnecessary column dimension, (batch,1) → (batch,).


## 13. TensorDataset and DataLoader: feeding the model 1,024 rows at a time

**Why do we need batches?** If the data has 100,000 rows, instead of loading every row onto the GPU at once, we can compute in chunks of 1,024 rows.

- `TensorDataset` keeps the `[product ID, customer ID]`, the `[2 means]`, and the `log target` of the same transaction together as one bundle.
- `DataLoader(..., batch_size=1024)` supplies the small chunks in sequence.
- `shuffle=True` shuffles the order of training batches, which helps training. For validation `False` is fine.

`torch.long` is the integer data type that embedding indices require, and `float32` is the floating-point data type commonly used in neural networks.

In [ ]:
def to_loader(cat, numeric, y, shuffle):  # bundle categorical IDs, numeric inputs, and targets in the same row order.
    cat_tensor = torch.tensor(cat, dtype=torch.long)  # integer ID Tensor that the embeddings must receive.
    numeric_tensor = torch.tensor(numeric, dtype=torch.float32)  # Target Encoding features as a floating-point Tensor.
    y_tensor = torch.tensor(y, dtype=torch.float32)  # the training target, log quantity, is also a float Tensor.
    dataset = TensorDataset(cat_tensor, numeric_tensor, y_tensor)  # bundle the three pieces of the same row into one example.
    loader = DataLoader(dataset, batch_size=1024, shuffle=shuffle)  # hands out up to 1024 examples each time.
    return loader  # return the batch supplier used in the training or evaluation loop.

train_loader = to_loader(cat_train, X_train, y_train, shuffle=True)  # training shuffles the rows randomly every epoch.
valid_loader = to_loader(cat_valid, X_valid, y_valid, shuffle=False)  # no reason to shuffle for validation.


## 14. Training function: predict → MSE → backpropagation → Adam update → validation → Early Stopping

**The neural-network training cycle** is as follows.

1. Predict with `net(cat,numeric)`.
2. Compute the prediction error with `mse_loss(prediction,target)`.
3. Compute the **gradient** of every Embedding/Linear parameter with `loss.backward()`.
4. Use that gradient to update the values with `optimizer.step()`.

Repeating this for every batch completes 1 epoch. After that, compute the Validation MSE with `net.eval()` and `torch.no_grad()`. Save the weights from the point where the MSE was best with `deepcopy(net.state_dict())`. When `stale` reaches 10, we consider that validation is no longer improving and stop training.

Why `reduction='sum'`? The last batch can have fewer than 1024 rows, so averaging the per-batch means again risks giving small and large batches equal weight. **Summing the squared errors of all transactions and dividing by the total number of transactions** gives the exact overall mean.

In [ ]:
def fit_net(hybrid, max_epochs=300, patience=10):  # choose hybrid, at most 300 epochs, stop after 10 epochs without improvement.
    torch.manual_seed(SEED)  # keep the initialization randomness the same when comparing (c) and (d).
    net = RetailNet(n_stock, n_cust, hybrid=hybrid).to(DEVICE)  # build the network and send it to the GPU/CPU.
    optimizer = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)  # Adam, learning rate 0.001, small L2 penalty.
    best_loss = float("inf")  # variable that stores the smallest validation log-MSE.
    best_epoch = 0  # number of the best epoch (counting from 1).
    best_state = None  # copy of the model parameters at the best-performing point.
    stale = 0  # number of consecutive epochs without validation improvement.

    for epoch in range(max_epochs):  # repeat a full pass over the data at most 300 times (early stopping usually ends it sooner).
        net.train()  # put the model in training mode.
        for cat, numeric, target in train_loader:  # fetch a batch of at most 1024 rows at a time.
            cat = cat.to(DEVICE)  # move the product/customer ID Tensor to the same device as the model.
            numeric = numeric.to(DEVICE)  # move the two numeric inputs to the same device too.
            target = target.to(DEVICE)  # move the log-quantity target to the same device too.
            optimizer.zero_grad()  # reset the gradients accumulated from the previous batch.
            prediction = net(cat, numeric)  # predict the log quantity with the current embeddings and weights.
            loss = nn.functional.mse_loss(prediction, target)  # compute the mean of (prediction - target)^2 over the batch.
            loss.backward()  # compute gradients for all trainable parameters via the chain rule.
            optimizer.step()  # Adam updates the network parameters using those gradients.

        net.eval()  # from here on, switch to the mode that only measures validation performance, without training.
        total_squared_error = 0.0  # variable holding the total squared error over all validation data.
        count = 0  # count all validation examples to compute the exact mean.
        with torch.no_grad():  # backpropagation is unnecessary for validation, so turn off gradient computation.
            for cat, numeric, target in valid_loader:  # iterate over the validation data batch by batch.
                cat = cat.to(DEVICE)  # move the validation product/customer IDs to the device.
                numeric = numeric.to(DEVICE)  # move the validation Target Encoding inputs to the device.
                target = target.to(DEVICE)  # move the validation log targets to the device too.
                pred = net(cat, numeric)  # predict on validation data not seen during training.
                batch_sum = nn.functional.mse_loss(pred, target, reduction="sum")  # compute the batch 'sum', not the mean.
                total_squared_error += batch_sum.item()  # accumulate per-batch error sums so each batch gets its proper weight.
                count += len(target)  # accumulate the number of examples in this batch.
        val_loss = total_squared_error / count  # exact validation MSE: total sum divided by total count.
        if val_loss < best_loss - 1e-6:  # did the validation MSE improve on the previous best by at least 1e-6?
            best_loss = val_loss  # store the newly recorded lowest MSE.
            best_epoch = epoch + 1  # also record which epoch it was.
            best_state = copy.deepcopy(net.state_dict())  # keep an 'independent copy' of all current weights.
            stale = 0  # it improved, so reset the consecutive no-improvement count to 0.
        else:  # if there was no improvement,
            stale += 1  # increase the consecutive no-improvement count by one.
        if stale >= patience:  # has the consecutive no-improvement count reached 10?
            break  # stop without further training: Early Stopping.

    net.load_state_dict(best_state)  # restore the 'weights with the best validation performance', not the last weights.
    print(f"hybrid={hybrid}, epochs={epoch+1}, best epoch={best_epoch}, best validation log-MSE={best_loss:.6f}")  # print a training summary.
    return net  # return the best model after training.


## 15. Predicting Test or other data with the trained network

**Keep training and prediction separate.** `predict_net` does not change the weights. `@torch.no_grad()` ensures no gradients are computed here.

`dummy_y` is just zeros filled in because the existing `to_loader` requires three inputs. The actual prediction is computed only from the two inputs `cat` and `numeric`. `np.concatenate` joins the length-1024 arrays produced for each batch into one long prediction array.

In [ ]:
@torch.no_grad()  # no gradients are computed anywhere in the function below. It is for prediction only, not training.
def predict_net(net, cat, numeric):  # takes a trained network, integer IDs, and numeric inputs, and produces log-quantity predictions.
    net.eval()  # switch to evaluation mode.
    predictions = []  # empty list to collect per-batch predictions in order.
    dummy_y = np.zeros(len(cat), dtype=np.float32)  # fake targets to match the DataLoader format; not used for prediction.
    loader = to_loader(cat, numeric, dummy_y, shuffle=False)  # supplies the data to predict in chunks of 1024 rows.
    for cat_batch, num_batch, _ in loader:  # use only each batch's ID/numeric inputs and discard the fake targets.
        cat_batch = cat_batch.to(DEVICE)  # move the IDs to the model's device.
        num_batch = num_batch.to(DEVICE)  # move the numeric inputs to the model's device too.
        batch_prediction = net(cat_batch, num_batch)  # predict the log quantity for one batch.
        predictions.append(batch_prediction.cpu().numpy())  # store the result as a NumPy array on the CPU.
    return np.concatenate(predictions)  # join the per-batch arrays into one long 1-D array in the original row order.


## 16. Training models (c) and (d) under the same conditions

These two lines run the experiments for **(c)** and **(d)**.

- `hybrid=False` → (c), which uses only the product/customer embeddings as input.
- `hybrid=True` → (d), which uses the product/customer embeddings + 2 Target Encodings.

The two experiments use the same train/validation split, log targets, batch size, base learning rate, and Early Stopping rule, so they are comparable. However, the input dimension and number of parameters differ, so we must not conclude that any performance difference is due to Target Encoding alone.

In [ ]:
embedding_model = fit_net(hybrid=False)  # (c) train an MLP using only the 8-D product + 8-D customer embeddings.
hybrid_model = fit_net(hybrid=True)  # (d) train an MLP that adds the 2 Target Encodings to the embeddings above.


## 17. (e) Test performance: report both in log units and in original Quantity units

**Now we use Test.** After choosing `alpha` and the best epoch on Validation, generate Test predictions for the three models.

1. `pred_log=np.maximum(...,0)` → clip log predictions at 0 or above so quantity predictions are never negative.
2. `np.expm1(pred_log)` → recover the *purchase quantity* units via $\hat y=e^{\hat t}-1$.
3. `log_MSE` → the mean of $(\log(1+y)-\hat t)^2$.
4. `RMSLE` → `sqrt(log_MSE)`.
5. `MAE` → mean absolute error in quantity units.
6. `RMSE` → square root of the mean squared error in quantity units.

**Note:** For all four metrics above, smaller is better. These results cannot be known before running on the actual `retail.csv` file. MSE in log space and RMSE in quantity space answer different questions.

In [ ]:
# Note: up to the point this function is called, Test has not been used to choose alpha, epochs, or networks.
def evaluate(name, pred_log):  # name is the model name, pred_log is the log(1+quantity) prediction.
    pred_log = np.maximum(pred_log, 0.0)  # clip log predictions at 0 or above so quantities are never negative.
    pred_qty = np.expm1(pred_log)  # recover original quantity units with exp(t)-1, the inverse of log(1+y).
    log_mse = np.mean((y_test - pred_log) ** 2)  # log-scale mean squared error: tied to the training criterion.
    rmsle = np.sqrt(log_mse)  # square root of the log MSE: smaller is better.
    mae = np.mean(np.abs(true_qty - pred_qty))  # on average, how many units off in the original Quantity scale?
    rmse = np.sqrt(np.mean((true_qty - pred_qty) ** 2))  # RMSE, which penalizes large quantity errors more heavily.
    return {"model": name, "log_MSE": log_mse, "RMSLE": rmsle, "MAE": mae, "RMSE": rmse}  # one row of the table.

A_test = np.column_stack([np.ones(len(X_test)), X_test])  # also prepend the intercept column of 1s to Ridge's Test input.
ridge_pred = A_test @ beta_ridge  # (b) test log predictions of the hand-implemented Ridge.
embed_pred = predict_net(embedding_model, cat_test, X_test)  # (c) test predictions of the embedding-only MLP.
hybrid_pred = predict_net(hybrid_model, cat_test, X_test)  # (d) test predictions of the Hybrid MLP.

rows = [  # list collecting each model's evaluation results into one table.
    evaluate("constant baseline", np.full(len(test), y_train.mean())),  # reference point: always predicts just the mean training log target.
    evaluate("target encoding + ridge", ridge_pred),  # (b) Target Encoding + Ridge.
    evaluate("two embeddings + MLP", embed_pred),  # (c) Embedding + MLP.
    evaluate("target encoding + embeddings + MLP", hybrid_pred),  # (d) Hybrid MLP.
]
results = pd.DataFrame(rows)  # turn each model's log-MSE, RMSLE, MAE, RMSE into a pandas table.
print("\nTEST RESULTS (lower is better for all metrics)")  # explains how to read the results table.
print(results.to_string(index=False, float_format=lambda v: f"{v:.6f}"))  # print each model's metrics to 6 decimal places.
results.to_csv("hw2_retail_results.csv", index=False)  # save to a CSV file so the submission results can be filled in.
print("Saved the results to hw2_retail_results.csv.")  # tell the user on screen that the file was created.


---
# Guide to writing the submission narrative

**After finishing, write it using the measured values in `hw2_retail_results.csv`.** Do not make up results now and fill them in.

**(b)** "We encoded StockCode and CustomerID using five-fold out-of-fold target means derived from the training data, transformed the heavy-tailed response with log1p, standardized the encoded predictors, and fit ridge regression. The regularization parameter was chosen on the validation set."

**(c)** "We trained an MLP using separate eight-dimensional embeddings for the product and customer categorical variables. We used log-MSE and early stopping based on validation performance."

**(d)** "The hybrid model concatenated both embeddings with two standardized target-encoded features. All models used the same data split and evaluation metrics."

**(e)** "[Insert the lowest test log-MSE model] performed best on test log-MSE. Embeddings may capture multidimensional differences and interactions that a single target mean cannot represent, although rare categories can produce unreliable representations. We cannot assume the hybrid model is superior without actual test results."

**Results table:** Fill in the four metrics **log_MSE / RMSLE / MAE / RMSE** for `constant baseline`, `target encoding + ridge`, `two embeddings + MLP`, and `target encoding + embeddings + MLP`.

**Interpretation tip:** After seeing the Test results, do not keep modifying the architecture or alpha to fit Test. If you want to try improvements, make those decisions on Validation and, if needed, do the final evaluation on a newly split, fully independent Test set.

## Actual final results (run on 2026-11-05, `results/hw2_retail_results.csv`)

| Model | log-MSE | RMSLE | MAE | RMSE |
|---|---:|---:|---:|---:|
| Constant baseline | 1.009571 | 1.004774 | 11.6955 | 307.1146 |
| Target encoding + ridge | 0.409972 | 0.640291 | 8.5201 | 305.9164 |
| Two embeddings + MLP | **0.281836** | **0.530882** | **7.3757** | **305.6237** |
| Target encoding + embeddings + MLP | 0.283321 | 0.532279 | 7.4918 | 306.2395 |

The Embedding-only MLP is best on this Test log-MSE and MAE. The baseline and ridge numbers come out identical on a rerun; the embedding/hybrid MLP numbers can shift in the fourth decimal place across PyTorch versions (same ranking). **Do not change the model settings again after checking the Test scores.**